## ESP32 CSI: Load 3 devices from one `test_*` folder and visualize

This notebook expects a folder like:

`wifi_data_set_fixed/id_person_01/label_00/test_01`

It loads all `.data` files in that folder (typically 3 ESP32 streams), parses them with `Parser` from `tools/csi_parser.py`, then plots:

1. CFR/ACH (mean amplitude vs subcarrier)
2. Mel-spectrograms from per-packet amplitude envelope

In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import numpy as np

# Make project root importable from experiments_notebooks/
PROJECT_ROOT = Path.cwd().resolve().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

from tools.csi_parser import Parser


def _hz_to_mel(hz: np.ndarray) -> np.ndarray:
    return 2595.0 * np.log10(1.0 + hz / 700.0)


def _mel_to_hz(mel: np.ndarray) -> np.ndarray:
    return 700.0 * (10.0 ** (mel / 2595.0) - 1.0)


def _mel_filterbank(sr: float, n_fft: int, n_mels: int = 40, fmin: float = 0.0, fmax: float | None = None) -> np.ndarray:
    if fmax is None:
        fmax = sr / 2.0

    mel_points = np.linspace(_hz_to_mel(np.array([fmin]))[0], _hz_to_mel(np.array([fmax]))[0], n_mels + 2)
    hz_points = _mel_to_hz(mel_points)
    bins = np.floor((n_fft + 1) * hz_points / sr).astype(int)

    fb = np.zeros((n_mels, n_fft // 2 + 1), dtype=np.float32)
    for m in range(1, n_mels + 1):
        left = bins[m - 1]
        center = bins[m]
        right = bins[m + 1]

        if center <= left:
            center = left + 1
        if right <= center:
            right = center + 1

        for k in range(left, center):
            fb[m - 1, k] = (k - left) / (center - left)
        for k in range(center, min(right, fb.shape[1])):
            fb[m - 1, k] = (right - k) / (right - center)

    return fb


def compute_mel_spectrogram(signal_1d: np.ndarray, sr_hz: float, n_fft: int = 64, hop_length: int = 8, n_mels: int = 40) -> np.ndarray:
    signal = np.asarray(signal_1d, dtype=np.float32)
    if signal.size < n_fft:
        pad = np.zeros(n_fft - signal.size, dtype=np.float32)
        signal = np.concatenate([signal, pad])

    n_frames = 1 + (signal.size - n_fft) // hop_length
    window = np.hanning(n_fft).astype(np.float32)

    stft_mag = []
    for i in range(n_frames):
        start = i * hop_length
        frame = signal[start : start + n_fft] * window
        spec = np.fft.rfft(frame)
        stft_mag.append(np.abs(spec) ** 2)

    power_spec = np.asarray(stft_mag, dtype=np.float32).T
    mel_fb = _mel_filterbank(sr=sr_hz, n_fft=n_fft, n_mels=n_mels)
    mel_power = mel_fb @ power_spec
    mel_db = 10.0 * np.log10(np.maximum(mel_power, 1e-10))
    return mel_db


def load_test_folder(test_folder: str | Path) -> dict[str, dict]:
    folder = Path(test_folder)
    if not folder.exists() or not folder.is_dir():
        raise ValueError(f"Invalid test folder: {folder}")

    data_files = sorted(folder.glob("*.data"))
    if len(data_files) == 0:
        raise ValueError(f"No .data files found in: {folder}")

    parsed: dict[str, dict] = {}
    for f in data_files:
        parsed[f.stem] = Parser(f).parse()
    
    print(parsed)

    return parsed


def plot_cfr_amplitude(parsed: dict[str, dict]) -> None:
    n = len(parsed)
    fig, axes = plt.subplots(1, n, figsize=(6 * n, 4), squeeze=False)
    axes = axes[0]

    for ax, (name, data) in zip(axes, parsed.items()):
        amp = data["amplitude"]
        mean_amp = amp.mean(axis=0)
        sub_idx = np.arange(mean_amp.shape[0])
        ax.plot(sub_idx, mean_amp, linewidth=2)
        ax.set_title(f"CFR/ACH mean amplitude\n{name}")
        ax.set_xlabel("Subcarrier index")
        ax.set_ylabel("Amplitude")
        ax.grid(True, alpha=0.3)

    plt.tight_layout()
    plt.show()


def plot_mel_spectrograms(parsed: dict[str, dict], packet_rate_hz: float = 20.0) -> None:
    n = len(parsed)
    fig, axes = plt.subplots(1, n, figsize=(6 * n, 4), squeeze=False)
    axes = axes[0]

    for ax, (name, data) in zip(axes, parsed.items()):
        # 1D signal from packet-wise average amplitude over all subcarriers
        packet_signal = data["amplitude"].mean(axis=1)
        mel_db = compute_mel_spectrogram(packet_signal, sr_hz=packet_rate_hz, n_fft=64, hop_length=8, n_mels=40)

        im = ax.imshow(mel_db, origin="lower", aspect="auto", cmap="magma")
        ax.set_title(f"Mel-spectrogram\n{name}")
        ax.set_xlabel("Frame")
        ax.set_ylabel("Mel bin")
        fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04, label="dB")

    plt.tight_layout()
    plt.show()

In [ ]:
# Example: one 5-second test folder with up to 3 ESP32 files
TEST_FOLDER = PROJECT_ROOT / "wifi_data_set_fixed" / "id_person_01" / "label_00" / "test_01"

parsed_data = load_test_folder(TEST_FOLDER)
print(f"Loaded devices: {list(parsed_data.keys())}")
for name, d in parsed_data.items():
    print(name, "packets=", d["packet_count"], "subcarriers=", d["subcarrier_count"])

plot_cfr_amplitude(parsed_data)
plot_mel_spectrograms(parsed_data, packet_rate_hz=20.0)